<a href="https://colab.research.google.com/github/rm237-rice/ELEC-576/blob/main/assignment_2_part_1_trainCifarStarterCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""Assignment_2_Part_1_Cifar10_vp1.ipynb

Purpose: Implement image classification nn the cifar10
dataset using a pytorch implementation of a CNN architecture (LeNet5)

Pseudocode:
1) Set Pytorch metada
- seed
- tensorboard output (logging)
- whether to transfer to gpu (cuda)

2) Import the data
- download the data
- create the pytorch datasets
    scaling
- create pytorch dataloaders
    transforms
    batch size

3) Define the model architecture, loss and optimizer

4) Define Test and Training loop
    - Train:
        a. get next batch
        b. forward pass through model
        c. calculate loss
        d. backward pass from loss (calculates the gradient for each parameter)
        e. optimizer: performs weight updates
        f. Calculate accuracy, other stats
    - Test:
        a. Calculate loss, accuracy, other stats

5) Perform Training over multiple epochs:
    Each epoch:
    - call train loop
    - call test loop
"""

'Assignment_2_Part_1_Cifar10_vp1.ipynb\n\nPurpose: Implement image classification nn the cifar10\ndataset using a pytorch implementation of a CNN architecture (LeNet5)\n\nPseudocode:\n1) Set Pytorch metada\n- seed\n- tensorboard output (logging)\n- whether to transfer to gpu (cuda)\n\n2) Import the data\n- download the data\n- create the pytorch datasets\n    scaling\n- create pytorch dataloaders\n    transforms\n    batch size\n\n3) Define the model architecture, loss and optimizer\n\n4) Define Test and Training loop\n    - Train:\n        a. get next batch\n        b. forward pass through model\n        c. calculate loss\n        d. backward pass from loss (calculates the gradient for each parameter)\n        e. optimizer: performs weight updates\n        f. Calculate accuracy, other stats\n    - Test:\n        a. Calculate loss, accuracy, other stats\n\n5) Perform Training over multiple epochs:\n    Each epoch:\n    - call train loop\n    - call test loop\n'

Step 1: Pytorch and Training Metadata



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.v2 as transformsV2
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import os
from pathlib import Path

#hyperparameters
batch_size = 128
epochs = 10
lr = 0.001
try_cuda = True
seed = 1000

# Architecture
num_classes = 10

#otherum
logging_interval = 10 # how many batches to wait before logging
logging_dir = None
grayscale = True

# setting up the logging

datetime_str = datetime.now().strftime('%b%d_%H-%M-%S')

if logging_dir is None:
    runs_dir = Path("./") / Path(f"runs/")
    runs_dir.mkdir(exist_ok = True)

    logging_dir = runs_dir / Path(f"{datetime_str}")

    logging_dir.mkdir(exist_ok = True)
    logging_dir = str(logging_dir.absolute())

writer = SummaryWriter(log_dir=logging_dir)

#deciding whether to send to the cpu or not if available
if torch.cuda.is_available() and try_cuda:
    cuda = True
    torch.cuda.manual_seed(seed)
else:
    cuda = False
    torch.manual_seed(seed)

Step 2: Data Setup

In [ ]:
# downloading the cifar10 dataset
from google.colab import files
uploaded = files.upload()

In [ ]:
# extracting the CIFAR10-1 zip file
import zipfile
import os

zip_path = 'CIFAR10-1.zip'
extract_path = './cifar10_extracted'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files structure:")
print(os.listdir(extract_path))

In [ ]:
import torch.nn.functional as F

num_classes = 10

# grayscale image transform
transform = transformsV2.Compose([
    transformsV2.Grayscale(num_output_channels=1),
    transformsV2.ToImage(),
    transformsV2.ToDtype(torch.float32, scale=True)
])

# target transform to convert integer labels to one-hot encoded tensors
target_transform = transformsV2.Lambda(
    lambda y: F.one_hot(torch.tensor(y), num_classes=num_classes).float()
)

In [ ]:
# Load dataset applying 'transform' on images and 'target_transform' on the labels
train_dataset = datasets.ImageFolder(
    root='./cifar10_extracted/CIFAR10/Train',
    transform=transform,
    target_transform=target_transform
)

test_dataset = datasets.ImageFolder(
    root='./cifar10_extracted/CIFAR10/Test',
    transform=transform,
    target_transform=target_transform
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
def check_data_loader_dim(loader):
    # Checking the dataset
    for images, labels in loader:
        print('Image batch dimensions:', images.shape)
        print('Image label dimensions:', labels.shape)
        break

check_data_loader_dim(train_loader)
check_data_loader_dim(test_loader)

In [ ]:
import matplotlib.pyplot as plt

# Display image and label.
train_features, train_labels = next(iter(train_loader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {label}")

Step 3: Creating the Model

In [ ]:
layer_1_n_filters = 6
layer_2_n_filters = 16
fc_1_n_nodes = 120
fc_2_n_nodes = 84
kernel_size = 5
verbose = False

# Input size is 28x28 (as shown in Step 2 data checks: [128, 1, 28, 28])
# Conv1 (padding 2, kernel 5): 28 -> 28
# MaxPool1 (kernel 2, stride 2): 28 -> 14
# Conv2 (no padding, kernel 5): 14 -> 10
# MaxPool2 (kernel 2, stride 2): 10 -> 5
fte_final_length = 5

if verbose:
    print(f"final length after feature extraction pipeline = {fte_final_length}")


class LeNet5(nn.Module):

    def __init__(self, num_classes, grayscale=False):
        super(LeNet5, self).__init__()

        self.grayscale = grayscale
        self.num_classes = num_classes

        if self.grayscale:
            in_channels = 1
        else:
            in_channels = 3

        self.conv1 = nn.Conv2d(in_channels, layer_1_n_filters, kernel_size, padding=2)
        self.tanh1 = nn.Tanh()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(layer_1_n_filters, layer_2_n_filters, kernel_size) # 6 input channels, 16 output channels (filter maps), 5x5 square convolution
        self.tanh2 = nn.Tanh()
        self.pool2 = nn.MaxPool2d(2)

        # Feature extractor: Convolutional and Pooling Layers
        self.features = nn.Sequential(
            self.conv1,
            self.tanh1,
            self.pool1,
            self.conv2,
            self.tanh2,
            self.pool2
        )

        self.fc1 = nn.Linear(fte_final_length * fte_final_length * layer_2_n_filters, fc_1_n_nodes)
        self.tanh3 = nn.Tanh()
        self.fc2 = nn.Linear(fc_1_n_nodes, fc_2_n_nodes)
        self.tanh4 = nn.Tanh()
        self.fc3 = nn.Linear(fc_2_n_nodes, num_classes)

        # Classifier: Flattened representation to fully connected layers
        self.classifier = nn.Sequential(
            self.fc1,
            self.tanh3,
            self.fc2,
            self.tanh4,
            self.fc3
        )

    def forward(self, x):
        # Send input through convolutional layers
        x = self.features(x)
        # Flatten the features representation for the fully connected layers
        x = torch.flatten(x, 1)
        # Send input through MLP/Classifier layers
        logits = self.classifier(x)
        probas = F.softmax(logits, dim=1)
        return logits, probas

# Instantiate the model
model = LeNet5(num_classes=num_classes, grayscale=grayscale)

if cuda:
    model.cuda()

# Initialize Adam Optimizer
optimizer = optim.Adam(model.parameters(), lr=lr)

Set Up Logging

In [ ]:
from torch.utils.tensorboard import SummaryWriter

# initializing the logger
writer = SummaryWriter()

In [ ]:
def log_tensor_stats(epoch, writer, tag, tensor):
    tensor = tensor.detach().float().cpu()

    writer.add_scalar(f"{tag}/min", tensor.min().item(), epoch)
    writer.add_scalar(f"{tag}/max", tensor.max().item(), epoch)
    writer.add_scalar(f"{tag}/mean", tensor.mean().item(), epoch)
    writer.add_scalar(f"{tag}/std", tensor.std().item(), epoch)

    writer.add_histogram(f"{tag}/histogram", tensor, epoch)

In [ ]:
# Dictionary that stores outputs from selected layers
activation_data = {}

def save_output(name):
    def hook(module, input, output):
        activation_data[name] = output.detach()
    return hook

# Net inputs / pre-activations
model.conv1.register_forward_hook(save_output("conv1_net"))
model.conv2.register_forward_hook(save_output("conv2_net"))

model.fc1.register_forward_hook(save_output("fc1"))
model.fc2.register_forward_hook(save_output("fc2"))
model.fc3.register_forward_hook(save_output("fc3"))

# Activations after Max-Pooling
model.pool1.register_forward_hook(save_output("pool1"))
model.pool2.register_forward_hook(save_output("pool2"))

# Activations after Tanh
model.tanh1.register_forward_hook(save_output("tanh1"))
model.tanh2.register_forward_hook(save_output("tanh2"))

model.tanh3.register_forward_hook(save_output("tanh3"))
model.tanh4.register_forward_hook(save_output("tanh4"))

In [ ]:
# Use the same batch every epoch for monitoring
monitor_data, monitor_target = next(iter(test_loader))

if cuda:
    monitor_data = monitor_data.cuda()
    monitor_target = monitor_target.cuda()

In [ ]:
def log_activations(epoch):

    model.eval()

    with torch.no_grad():
        _ = model(monitor_data)

    # Net inputs / pre-activations
    log_tensor_stats(epoch, writer, "NetInputs/conv1", activation_data["conv1_net"])
    log_tensor_stats(epoch, writer, "NetInputs/conv2", activation_data["conv2_net"])

    log_tensor_stats(epoch, writer, "NetInputs/fc1", activation_data["fc1_net"])
    log_tensor_stats(epoch, writer, "NetInputs/fc2", activation_data["fc2_net"])
    log_tensor_stats(epoch, writer, "NetInputs/fc3", activation_data["fc3_net"])

    # Max-pooling activations
    log_tensor_stats(epoch, writer, "Activations/MaxPool1", activation_data["pool1"])
    log_tensor_stats(epoch, writer, "Activations/MaxPool2", activation_data["pool2"])

    # Tanh activations
    log_tensor_stats(epoch, writer, "Activations/Tanh1", activation_data["tanh1"])
    log_tensor_stats(epoch, writer, "Activations/Tanh2", activation_data["tanh2"])
    log_tensor_stats(epoch, writer, "Activations/Tanh1", activation_data["tanh3"])
    log_tensor_stats(epoch, writer, "Activations/Tanh2", activation_data["tanh4"])

Step 4: Train/Test Loop

In [ ]:
# Defining the training loop

def train(epoch):
    model.train()
    running_loss = 0.
    last_loss = 0.

    criterion = nn.CrossEntropyLoss() # The choice of criterion depends on the output of the last layer of your network
    for batch_idx, (data, target) in enumerate(train_loader):
        # move data to GPU
        if cuda:
            data, target = data.cuda(), target.cuda()

        optimizer.zero_grad()
        logits,probas = model(data) # forward

        loss = criterion(probas, target)
        loss.backward()
        optimizer.step()

        # training loop and log metrics
        running_loss += loss.item()

        if batch_idx % logging_interval == 0:
            last_loss = running_loss / logging_interval # loss per batch
            tb_x = epoch * len(train_loader) + batch_idx + 1
            writer.add_scalar('Loss/train', last_loss, tb_x)
            print('Train set: Epoch: {} Batch: {} Loss: {:.4f}\n'.format(epoch, batch_idx + 1, last_loss))
            running_loss = 0. # reset running loss calculation

    # Log model parameters to TensorBoard at every epoch
    for name, param in model.named_parameters():
        if "weight" in name:
            tag = f"Weights/{name}"
        elif "bias" in name:
            tag = f"Biases/{name}"
        else:
            continue
        log_tensor_stats(epoch, writer, tag, param)

In [ ]:
# Defining the testing loop and log relevant metrics

def test(epoch):
    model.eval()
    test_loss = 0
    correct = 0
    # Use reduction='sum' to accumulate the absolute sum of losses across all samples
    criterion = nn.CrossEntropyLoss(reduction='sum')

    with torch.no_grad():
        for data, target in test_loader:
            # move data to GPU
            if cuda:
              data, target = data.cuda(), target.cuda()

            logits, probas = model(data)

            test_loss += criterion(probas, target).item()

            # Calculate the predicted label (highest probability) by taking the index of the max probability
            pred = probas.argmax(dim=1, keepdim=True)

            # Convert one-hot target to class indices
            target_indices = target.argmax(dim=1, keepdim=True)

            # Calculate the number of correct predictions
            correct += pred.eq(target_indices).sum().item()

        # Divide total accumulated loss by the total number of dataset samples
        test_loss /= len(test_loader.dataset)

        # Calculate the prediction accuracy
        test_accuracy = 100. * correct / len(test_loader.dataset)
        print('Test set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(test_loss, correct, len(test_loader.dataset), test_accuracy))

    # Log loss/test and accuracy/test to TensorBoard at every epoch
    n_iter = epoch * len(train_loader)
    writer.add_scalar('Loss/test', test_loss, n_iter)
    writer.add_scalar('Accuracy/test', test_accuracy, n_iter)

In [ ]:
# !pip install cuda-python
# !pip install tensorboard
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
# running test and training over epochs
for epoch in range(epochs):
    train(epoch)
    test(epoch)
writer.flush()

In [ ]:
writer.close()

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
"""
#https://stackoverflow.com/questions/55970686/tensorboard-not-found-as-magic-function-in-jupyter

#seems to be working in firefox when not working in Google Chrome when running in Colab
#https://stackoverflow.com/questions/64218755/getting-error-403-in-google-colab-with-tensorboard-with-firefox


# %load_ext tensorboard
# %tensorboard --logdir [dir]

"""